# Register Qwen3-VL as a Databricks Model Serving Endpoint

One-time (or occasional, if the model changes) infrastructure setup —
**not** part of the bundle-managed ingestion pipeline. Deploys
`Qwen/Qwen3-VL-4B-Instruct` (the Hugging Face Transformers release, *not*
the Ollama GGUF build used for local testing) as a custom, GPU-backed
Model Serving endpoint, so the RAG ingestion pipeline can call it the same
way it called every model in the OCR spike test — an OpenAI-style chat
completions API accepting image input.

**Prerequisites:**
- Attach this notebook to a cluster with a **GPU** — needed for the local
  smoke-test step below. A classic all-purpose cluster with a single A10
  node works fine (simpler, more transparent DBU-based pricing than
  serverless GPU compute); this is a one-time/occasional interactive setup
  task, so spin it up, run the notebook, and terminate it when done. This
  cluster is only for *running this notebook* — it is separate from the
  deployed serving endpoint's own GPU compute (step 4 below), which is
  Databricks-managed and billed independently (scale-to-zero).
- `mlflow>=3.12.0` and `databricks-sdk>=0.102.0` (installed below).

**A note on confidence.** This follows Databricks' documented pattern for
serving a custom chat/multimodal model via vLLM
([Serve custom LLMs with Custom Model Serving](https://docs.databricks.com/aws/en/machine-learning/model-serving/serve-custom-llms)),
cross-checked against a real worked example using the same pattern. The
overall flow (download weights → smoke-test locally with vLLM → log to
MLflow with a `task`/`entrypoint` metadata pair → register to Unity
Catalog → create a GPU serving endpoint) is well-confirmed. The one piece
reconstructed rather than copied verbatim from an official example is how
the `entrypoint` command's `--model` path resolves once it's an MLflow
artifact rather than a local directory — that's why the notebook runs an
**identical** vLLM launch command locally first (cells below) before
logging it: if the path convention is wrong, it fails fast in the cheap
local test rather than after a 10+ minute endpoint deployment.


In [ ]:
%pip install "mlflow>=3.12.0" "databricks-sdk>=0.102.0" "huggingface_hub>=0.24" "vllm>=0.11.0" pdfplumber
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "eliao")
dbutils.widgets.text("schema", "wnv_demo")
dbutils.widgets.text("model_name", "qwen3_vl_ocr")
dbutils.widgets.text("hf_model_id", "Qwen/Qwen3-VL-4B-Instruct")
dbutils.widgets.text("endpoint_name", "qwen3-vl-ocr")
# GPU_MEDIUM = 1x A10 (24GB) -- Databricks' own docs list this as the
# default tier for general inference, and their worked example for a
# *smaller* vision model (Qwen2.5-VL-3B) uses A10, not T4. Vision models
# need more headroom than a same-size text model: page-image inputs
# decode into a lot of vision tokens, which consume KV-cache memory on
# top of the vision encoder itself. GPU_SMALL (T4, 16GB) was the initial
# assumption in the design doc but wasn't actually validated -- corrected
# here.
dbutils.widgets.dropdown("workload_type", "GPU_MEDIUM", ["GPU_SMALL", "GPU_MEDIUM", "GPU_LARGE"])
# Only needed for the end-to-end validation cell at the bottom.
dbutils.widgets.text(
    "test_pdf_volume_path",
    "/Volumes/eliao/wnv_demo/documents/WNV-Outbreak-Communications-Toolkit-2025_508c.pdf",
)
dbutils.widgets.text("test_page", "5")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
model_name = dbutils.widgets.get("model_name").strip()
hf_model_id = dbutils.widgets.get("hf_model_id").strip()
endpoint_name = dbutils.widgets.get("endpoint_name").strip()
workload_type_str = dbutils.widgets.get("workload_type").strip()

uc_model_name = f"{catalog}.{schema}.{model_name}"
print(f"UC model:       {uc_model_name}")
print(f"Endpoint:       {endpoint_name}")
print(f"Source model:   {hf_model_id}")
print(f"GPU workload:   {workload_type_str}")

## 1. Download model weights

Downloads the actual Hugging Face Transformers release (safetensors) to
local (non-`/Workspace`) storage — large model weights don't belong on the
workspace filesystem, and this is a genuinely different artifact format
from the Ollama GGUF build already on your machine; that one is for
llama.cpp-style local inference, not compatible with the serving path here.


In [ ]:
import tempfile
from pathlib import Path

from huggingface_hub import snapshot_download

local_model_dir = Path(tempfile.mkdtemp()) / "model"
snapshot_download(repo_id=hf_model_id, local_dir=str(local_model_dir))
print(f"Downloaded {hf_model_id} to {local_model_dir}")

## 2. Smoke-test locally with vLLM

Launches the *exact* command Model Serving will run later (same flags,
different port), so any problem with the model path or vLLM flags surfaces
here — a couple of minutes — rather than after a full endpoint deployment.
This is also Databricks' own documented recommended step before logging.


In [ ]:
# Some hosts (particularly certain regulated/enterprise cloud node
# images) enforce OpenSSL FIPS mode at the OS level -- if so, several of
# vLLM's dependencies hard-crash with "FATAL FIPS SELFTEST FAILURE"
# (a process abort, not a catchable Python exception).
import subprocess

_fips = subprocess.run(
    ["cat", "/proc/sys/crypto/fips_enabled"], capture_output=True, text=True
)
if _fips.returncode != 0:
    fips_enabled = "not present on this host (no kernel-level FIPS flag)"
else:
    fips_enabled = _fips.stdout.strip()
print(f"FIPS mode (/proc/sys/crypto/fips_enabled): {fips_enabled}")
if fips_enabled == "1":
    print(
        "WARNING: this host enforces FIPS mode at the OS level -- likely "
        "cause of a 'FATAL FIPS SELFTEST FAILURE' crash below. This is a "
        "cluster/node-image choice, not fixable via pip."
    )
else:
    print(
        "No host-level FIPS enforcement detected. If the next cells still "
        "hit FATAL FIPS SELFTEST FAILURE, it's a specific dependency's "
        "bundled OpenSSL build misbehaving, independent of the host -- "
        "see the bisection cell below to identify which one."
    )

### If it crashes: bisect which import is the cause

The FIPS abort happens at C-library level during import, before any Python traceback can form -- the process output alone can't tell us which dependency did it. This cell imports vLLM's heavy/crypto-touching dependencies one at a time, each in its own fresh subprocess, so whichever one crashes identifies itself. Skip this cell if the FIPS check above found no host-level enforcement and you want to just try the smoke test first -- but run this if that smoke test hits the same crash, rather than guessing at a fix.


In [ ]:
import subprocess
import sys

# Rough dependency order vLLM pulls in -- crypto/networking libraries
# first (the usual suspects for bundled-OpenSSL issues), then vLLM
# itself last since it transitively imports most of the others anyway.
candidates = ["ssl", "cryptography", "grpc", "ray", "torch", "vllm"]

for module in candidates:
    result = subprocess.run(
        [sys.executable, "-c", f"import {module}; print('IMPORT_OK')"],
        capture_output=True,
        text=True,
        timeout=180,
    )
    ok = result.returncode == 0 and "IMPORT_OK" in result.stdout
    print(f"{module:15s} {'OK' if ok else f'FAILED (rc={result.returncode})'}")
    if not ok:
        combined = (result.stdout + result.stderr).strip()
        print(f"  --> {combined[-1000:]}")
        if "FIPS SELFTEST" in combined:
            print(f"  *** Found it: `{module}` triggers the FIPS self-test failure. ***")
            break

# All bare imports passed -- the crash is in runtime behavior, not
# import time. Test the two most likely runtime-only triggers: an
# actual TLS network call (huggingface.co reachability check), and
# ray.init() (which actually starts Ray's runtime, unlike `import ray`).
runtime_checks = {
    "https request": "import requests; requests.get('https://huggingface.co', timeout=10); print('IMPORT_OK')",
    "ray.init()": "import ray; ray.init(ignore_reinit_error=True); print('IMPORT_OK')",
}
for label, snippet in runtime_checks.items():
    result = subprocess.run(
        [sys.executable, "-c", snippet], capture_output=True, text=True, timeout=60
    )
    ok = result.returncode == 0 and "IMPORT_OK" in result.stdout
    print(f"{label:15s} {'OK' if ok else f'FAILED (rc={result.returncode})'}")
    if not ok:
        combined = (result.stdout + result.stderr).strip()
        print(f"  --> {combined[-1000:]}")

# Nothing above actually touched the GPU -- import torch doesn't
# initialize CUDA (that's lazy), and neither ray.init() nor a network
# call does either. Starting the real vLLM engine is the only thing
# tested so far that creates a CUDA context. Test that in isolation.
gpu_check = (
    "import torch; x = torch.zeros(1).cuda(); print(x.device); print('IMPORT_OK')"
)
result = subprocess.run(
    [sys.executable, "-c", gpu_check], capture_output=True, text=True, timeout=120
)
ok = result.returncode == 0 and "IMPORT_OK" in result.stdout
print(f"{'cuda init':15s} {'OK' if ok else f'FAILED (rc={result.returncode})'}")
if not ok:
    combined = (result.stdout + result.stderr).strip()
    print(f"  --> {combined[-1500:]}")
    if "FIPS SELFTEST" in combined:
        print(
            "  *** Found it: CUDA/GPU driver initialization triggers the "
            "FIPS self-test, not any Python library. Likely something in "
            "the NVIDIA driver/CUDA stack on this node image links a "
            "FIPS-mode OpenSSL (e.g. for driver telemetry/licensing). This "
            "would point at the node/runtime image itself, not a pip "
            "dependency -- worth trying a different GPU node type or "
            "escalating to your Databricks admin with this exact repro."
        )

In [ ]:
import json
import subprocess
import time
import urllib.request

LOCAL_TEST_PORT = 8000

# Model is already downloaded locally -- no reason to touch the network
# at load time. Also rules out a TLS handshake as the crash trigger.
import os

offline_env = {**os.environ, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}

vllm_proc = subprocess.Popen(
    [
        "python", "-u", "-m", "vllm.entrypoints.openai.api_server",
        "--model", str(local_model_dir),
        "--served-model-name", "qwen3-vl-ocr",
        "--host", "0.0.0.0",
        "--port", str(LOCAL_TEST_PORT),
        "--dtype", "float16",
        "--max-model-len", "8192",
        "--gpu-memory-utilization", "0.85",
        "--limit-mm-per-prompt", "image=1",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=offline_env,
)

ready = False
for _ in range(120):  # up to ~10 min for first-time weight load
    if vllm_proc.poll() is not None:
        break  # process exited early -- something is wrong
    try:
        urllib.request.urlopen(f"http://localhost:{LOCAL_TEST_PORT}/v1/models", timeout=2)
        ready = True
        break
    except Exception:
        time.sleep(5)

if not ready:
    output = vllm_proc.stdout.read() if vllm_proc.stdout else ""
    vllm_proc.terminate()
    raise RuntimeError(f"vLLM server did not become ready. Full output:\n{output}")

print("Local vLLM server is up.")

In [ ]:
payload = {
    "model": "qwen3-vl-ocr",
    "messages": [{"role": "user", "content": "Reply with exactly: OK"}],
    "max_tokens": 10,
    "temperature": 0.0,
}
req = urllib.request.Request(
    f"http://localhost:{LOCAL_TEST_PORT}/v1/chat/completions",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req, timeout=60) as resp:
    result = json.loads(resp.read().decode())
print(result["choices"][0]["message"]["content"])

vllm_proc.terminate()
vllm_proc.wait(timeout=30)
print("Local smoke test passed; local vLLM server stopped.")

## 3. Log to MLflow

Uses Databricks' `task: llm/v1/chat` + `entrypoint` pattern: Model Serving
runs the `entrypoint` shell command directly as the actual inference
server for this task type — it does **not** call the Python model's
`predict()`. The `python_model=` argument is a required-but-unused
placeholder for that reason. The entrypoint command below is identical to
the one just smoke-tested above, except the model path (now the artifact
key, resolved against the MLflow model's artifacts folder at serving time)
and the port (Model Serving requires 8080).


In [ ]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

SERVING_PORT = 8080  # Required by Model Serving for this task type.
ARTIFACT_KEY = "model_dir"


def build_entrypoint(port: int) -> str:
    # HF_HUB_OFFLINE/TRANSFORMERS_OFFLINE: model is already the logged
    # artifact, no reason to reach the network at load time -- also
    # matches the local smoke test above.
    return (
        "HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 "
        "python -u -m vllm.entrypoints.openai.api_server "
        f"--model {ARTIFACT_KEY} --served-model-name qwen3-vl-ocr "
        f"--host 0.0.0.0 --port {port} "
        "--dtype float16 --max-model-len 8192 "
        "--gpu-memory-utilization 0.85 "
        "--limit-mm-per-prompt image=1"
    )


class _VLLMEntrypointModel(mlflow.pyfunc.PythonModel):
    """Unused placeholder -- see markdown above."""

    def predict(self, context, model_input):
        raise RuntimeError(
            "Not used -- Model Serving runs `entrypoint` directly for "
            "llm/v1/chat custom models."
        )


model_info = mlflow.pyfunc.log_model(
    name=model_name,
    python_model=_VLLMEntrypointModel(),
    artifacts={ARTIFACT_KEY: str(local_model_dir)},
    metadata={
        "task": "llm/v1/chat",
        "entrypoint": build_entrypoint(SERVING_PORT),
    },
    extra_pip_requirements=["mlflow==3.12.0", "vllm>=0.11.0"],
)
print(f"Logged: {model_info.model_uri}")

In [ ]:
model_version = mlflow.register_model(model_uri=model_info.model_uri, name=uc_model_name)
print(f"Registered: {uc_model_name} version {model_version.version}")

## 4. Create the GPU serving endpoint

`scale_to_zero_enabled=True` — matches the design decision that this
endpoint should cost nothing between ingestion runs (the ingestion job is
scheduled/manually-triggered, not continuous). First call after being idle
will have a cold-start delay while the endpoint reloads the model.


In [ ]:
from datetime import timedelta

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)

w = WorkspaceClient()

config = EndpointCoreConfigInput(
    served_entities=[
        ServedEntityInput(
            entity_name=uc_model_name,
            entity_version=str(model_version.version),
            workload_type=getattr(ServingModelWorkloadType, workload_type_str),
            workload_size="Small",
            scale_to_zero_enabled=True,
        )
    ]
)

print(f"Creating endpoint '{endpoint_name}' -- can take 10+ minutes for a first deploy.")
w.serving_endpoints.create_and_wait(name=endpoint_name, config=config, timeout=timedelta(minutes=45))
print("Endpoint ready.")

## 5. Validate end-to-end

Same request shape validated in the OCR spike test (`notebooks/rag_ocr_spike.ipynb`)
— confirms this endpoint is a drop-in for `WNV_VISION_ENDPOINT` with no
other code changes needed.


In [ ]:
import base64
import io
from urllib import request as urlrequest

import pdfplumber

test_pdf_path = dbutils.widgets.get("test_pdf_volume_path").strip()
test_page_num = int(dbutils.widgets.get("test_page").strip())

context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
databricks_host = context.apiUrl().get().removeprefix("https://").removesuffix("/")
databricks_token = context.apiToken().get()

with pdfplumber.open(test_pdf_path) as pdf:
    page = pdf.pages[test_page_num - 1]
    image = page.to_image(resolution=300)
    buf = io.BytesIO()
    image.original.save(buf, format="PNG")
    image_bytes = buf.getvalue()

b64 = base64.b64encode(image_bytes).decode()
payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Extract all text from this document page as clean markdown."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            ],
        }
    ],
    "max_tokens": 2000,
    "temperature": 0.0,
}
url = f"https://{databricks_host}/serving-endpoints/{endpoint_name}/invocations"
req = urlrequest.Request(
    url,
    data=json.dumps(payload).encode(),
    headers={
        "Authorization": f"Bearer {databricks_token}",
        "Content-Type": "application/json",
    },
    method="POST",
)
with urlrequest.urlopen(req, timeout=120) as resp:
    result = json.loads(resp.read().decode())

print(result["choices"][0]["message"]["content"])

## Done

Set `WNV_VISION_ENDPOINT` to the value of `endpoint_name` above (default
`qwen3-vl-ocr`) wherever the ingestion pipeline configures it. The endpoint
is scale-to-zero, so the first call after being idle will be slow (model
reload) — expected, not a bug.
